In [2]:
import re
import os
import json

# Define the correct action formats
ACTION_FORMATS = {
    "walk": 1, "run": 1, "walktowards": 1, "walkforward": 0,
    "turnleft": 0, "turnright": 0, "sit": 1, "standup": 0,
    "grab": 1, "open": 1, "close": 1, "put": 2, "putin": 2,
    "switchon": 1, "switchoff": 1, "drink": 1, "touch": 1, "lookat": 1
}

# Loads the room-object mapping for the specified environment
def load_environment_data(json_file, env_number):
    with open(json_file, "r", encoding="utf-8") as f:
        data = json.load(f)

    env_key = f"env_{env_number}"
    if env_key not in data:
        raise ValueError(f"Environment {env_number} not found in JSON file.")
    
    return data[env_key]

# Checks whether each action in the post-processed file follows the required format
def format_check(post_file, env_data):
    if not os.path.isfile(post_file):
        print(f"Post-processed file not found: {post_file}")
        return

    #with open(post_file, "r", encoding="utf-8") as f:
    with open(post_file, "r", encoding="utf-8", errors="replace") as f:
        post_lines = f.readlines()

    total_lines = 0
    invalid_lines = 0
    missing_actions = 0
    unknown_actions = 0
    missing_objects = 0
    total_wrong_lines = 0
    incorrect_objects = 0
    total_checked_objects = 0

    print("\n🔍 **Format Check Report:**")

    for i, line in enumerate(post_lines, start=1):
        line = line.strip()

        if not line:  # Skip empty lines
            continue
        
        # count
        total_lines += 1
        mistake_count = 0
        
        # Detect missing actions
        if line == "[]":  
            print(f"❌ Missing action detected at line {total_lines}: {line}")
            missing_actions += 1
            mistake_count += 1
            total_wrong_lines += 1
            continue
            
        # Detect missing objects
        if "<>" in line:
            print(f"❌ Missing object at line {total_lines}: {line}")
            missing_objects += 1
            mistake_count += 1
        
        room_match = re.search(r"\(([^()]*)\)\s*$", line)

        if not room_match:
            print(f"⚠️ Warning: No room detected at line {total_lines}: {line}")
        room = room_match.group(1).lower()
        
        # Extract objects from angle brackets <>
        objects = re.findall(r"<(.*?)>", line)
        
        objects_clean = [obj for obj in objects if obj != ""]  # Remove "<>" placeholders

        valid_rooms = {"bathroom", "bedroom", "kitchen", "livingroom"}
        for obj in objects_clean:
            total_checked_objects += 1
            if obj in valid_rooms:
                continue
            if obj not in env_data.get(room, []):  # Check if object exists in the correct room
                print(f"❌ Incorrect object at line {total_lines}: {line} (Object '{obj}' does not belong in '{room}')")
                incorrect_objects += 1
                mistake_count += 1
        
        # Extract action from brackets []
        action_match = re.match(r"\[(.*?)\]", line)
        action = action_match.group(1) if action_match else None
        
        # Validate action format
        if action not in ACTION_FORMATS:
            print(f"❌ Unknown action at line {total_lines}: {line}")
            unknown_actions += 1
            total_wrong_lines += 1
            continue
             
        # Validate object count
        expected_objects = ACTION_FORMATS[action]
        if len(objects) != expected_objects:
            print(f"❌ Incorrect format at line {total_lines}: {line} (Expected {expected_objects} object(s), Found {len(objects)})")
            invalid_lines += 1
            mistake_count += 1
        
        if mistake_count > 0:
            total_wrong_lines += 1
        
        
    # Summary report
    evaluation_results = {
        "file": post_file,
        "total_lines": total_lines,
        "total_wrong_lines": total_wrong_lines,
        "total_wrong_line_percentage": (total_wrong_lines / total_lines) * 100 if total_lines else 0,
        "invalid_format_lines": invalid_lines,
        "invalid_format_percentage": (invalid_lines / total_lines) * 100 if total_lines else 0,
        "missing_actions": missing_actions,
        "missing_action_percentage": (missing_actions / total_lines) * 100 if total_lines else 0,
        "unknown_actions": unknown_actions,
        "unknown_action_percentage": (unknown_actions / total_lines) * 100 if total_lines else 0,
        "lines_with_missing_objects": missing_objects,
        "missing_object_percentage": (missing_objects / total_lines) * 100 if total_lines else 0,
        "total_objects_checked": total_checked_objects,
        "incorrect_objects": incorrect_objects,
        "incorrect_object_percentage": (incorrect_objects / total_checked_objects) * 100 if total_checked_objects else 0
    }
    
    return evaluation_results

In [3]:
def run_evaluation(post_processed_folder):
    
    # Path to the JSON file
    json_file = "room_objects.json"
    
    # Ensure the folder exists
    if not os.path.exists(post_processed_folder):
        print(f"🚨 Folder not found: {post_processed_folder}")
    else:
        evaluation_summary = []

        # Loop through all files in the folder
        for filename in sorted(os.listdir(post_processed_folder)):
            if filename.endswith(".txt"):
                # Extract env ID from filename (e.g., env_3_Monday.txt → 3)
                env_match = re.search(r'env_(\d+)_', filename)
                if not env_match:
                    print(f"⚠️ Skipping file without valid env ID: {filename}")
                    continue

                env_number = int(env_match.group(1))
                env_data = load_environment_data(json_file, env_number)

                # Perform the evaluation
                post_processed_file = os.path.join(post_processed_folder, filename)
                evaluation_results = format_check(post_processed_file, env_data)

                if evaluation_results:
                    evaluation_summary.append(evaluation_results)


        # Print overall summary
        print("\n📊 **Overall Summary for All Files:**")
        for result in evaluation_summary:
            print(f"\n📂 File: {result['file'].replace(os.sep, '/')}")
            print(f"Total lines checked: {result['total_lines']}")
            print(f"Total wrong lines: {result['total_wrong_lines']} ({result['total_wrong_line_percentage']:.2f}%)")
            print(f"Invalid format lines: {result['invalid_format_lines']} ({result['invalid_format_percentage']:.2f}%)")
            print(f"Missing actions: {result['missing_actions']} ({result['missing_action_percentage']:.2f}%)")
            print(f"Unknown actions: {result['unknown_actions']} ({result['unknown_action_percentage']:.2f}%)")
            print(f"Lines with missing objects: {result['lines_with_missing_objects']} ({result['missing_object_percentage']:.2f}%)")
            print(f"Total objects checked: {result['total_objects_checked']}")
            print(f"Incorrect objects found: {result['incorrect_objects']} ({result['incorrect_object_percentage']:.2f}%)")


In [4]:
# =========================
# Configuration (User-editable)
# =========================

# Choose which step to evaluate:
# data/step_5_data
# data/step_6_data
# data/step_7_data
post_processed_folder = "data/step_7_data"

run_evaluation(post_processed_folder)


🔍 **Format Check Report:**
❌ Missing object at line 31: [put] <> <rug> (06:55 - 06:56) (bedroom)
❌ Missing object at line 38: [sit] <> (07:01 - 07:01) (bedroom)
❌ Missing object at line 51: [lookat] <> (07:12 - 07:12) (bedroom)
❌ Missing object at line 62: [sit] <> (07:21 - 07:21) (bedroom)
❌ Missing object at line 103: [put] <> <kitchencounter> (07:57 - 07:58) (kitchen)
❌ Missing action detected at line 139: []
❌ Missing object at line 147: [put] <> <dishbowl> (18:37 - 18:38) (kitchen)
❌ Missing action detected at line 332: []
❌ Missing action detected at line 333: []

🔍 **Format Check Report:**
❌ Missing object at line 29: [put] <> <rug> (7:22 - 7:23) (bedroom)
❌ Missing object at line 254: [put] <> <kitchencounter> (19:11 - 19:12) (kitchen)
❌ Missing object at line 256: [put] <> <kitchencounter> (19:15 - 19:15) (kitchen)
❌ Missing object at line 298: [put] <> <nightstand> (21:20 - 21:20) (bedroom)
❌ Missing object at line 303: [put] <> <coffeetable> (21:30 - 21:30) (bedroom)
❌ Miss